[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/06_RQ4_Foundation_Models.ipynb)

# Notebook 06 — RQ4: Foundation Model Architecture

> **RQ4.** How does a foundation-model approach (zero-shot and few-shot prompting, and retrieval-augmented summarization) compare with the best fine-tuned model in accuracy, cost, and latency, and which architecture should a business adopt?

**Design**
1. **Sample:** 2,000 test reviews: 1,000 IMDb (official test) and 1,000 Amazon (test), each balanced by class, seed 42.
2. **Prompting:** zero-shot, 4-shot and 8-shot (balanced, domain-matched examples taken from the training data, never from the test data). Accuracy, macro F1, latency and cost per 1,000 reviews are recorded.
3. **Comparison:** the best fine-tuned model from RQ2 scores the same reviews: IMDb DistilBERT for movie reviews and Amazon-adapted DistilBERT for product reviews. Paired McNemar tests compare it with each prompting setting.
4. **RAG summaries:** for 30 Amazon products, the most relevant praise and complaint reviews are retrieved with sentence embeddings (Lewis et al., 2020), and the language model writes cited theme summaries. These are rated for faithfulness and usefulness (3-point rubric) by the author and one peer reviewer.
5. **Break-even:** the monthly review volume at which self-hosting a fine-tuned model becomes cheaper than paying per call.

### Choosing the language model
| `PROVIDER` | What runs | Needs |
|---|---|---|
| `"hf_local"` (default) | Open instruction-tuned model on the Colab GPU (default `Qwen/Qwen2.5-1.5B-Instruct`) | Nothing; free |
| `"anthropic"` | Claude via API (default `claude-haiku-5-5`) | `ANTHROPIC_API_KEY` in Colab **Secrets** (🔑 icon) and current prices |
| `"openai"` | OpenAI via API | `OPENAI_API_KEY` in Colab Secrets, a model name and current prices |

Record the model name, version and run date in the report, because hosted models and prices change.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

In [ ]:
!pip -q install sentence-transformers anthropic openai
import importlib, json, time, re, math, random
import cvi_utils as U
importlib.reload(U); U.set_project(PROJECT_DIR)
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
U.plot_style(); U.set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("Device:", DEVICE)

## 1. Configuration

In [ ]:
PROVIDER = "hf_local"                         # "hf_local" | "anthropic" | "openai"
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"       # e.g. "Qwen/Qwen2.5-3B-Instruct" for higher accuracy (slower)
ANTHROPIC_MODEL = "claude-haiku-5-5"
OPENAI_MODEL = ""                             # fill in the model name you use

# USD per million tokens: fill in from the provider's pricing page ON THE DAY YOU RUN, and cite it in the report
API_PRICES = {"anthropic": dict(input=None, output=None), "openai": dict(input=None, output=None)}

N_PER_DOMAIN = 1000                           # 2 x 1,000 = 2,000 reviews
SETTINGS = {"zero_shot": 0, "few_shot_4": 4, "few_shot_8": 8}
MAX_REVIEW_WORDS, MAX_EXAMPLE_WORDS = 350, 120
N_PRODUCTS, TOP_K = 30, 8
MAX_LEN_BERT = 512

if U.QUICK:
    N_PER_DOMAIN, N_PRODUCTS = 20, 2
    SETTINGS = {"zero_shot": 0, "few_shot_4": 4}

RES = U.P("outputs", "results"); MOD2 = U.P("models", "rq2")
COSTS = json.load(open(f"{RES}/cost_assumptions.json"))
MODEL_TAG = {"hf_local": HF_MODEL, "anthropic": ANTHROPIC_MODEL, "openai": OPENAI_MODEL}[PROVIDER]
PRED_CSV = f"{RES}/rq4_llm_preds_{PROVIDER}_{MODEL_TAG.replace('/', '_')}.csv"
print("Provider:", PROVIDER, "| model:", MODEL_TAG, "| run date:", time.strftime("%Y-%m-%d"))

## 2. Evaluation sample and few-shot examples

In [ ]:
imdb, amz = U.load_imdb(), U.load_amazon()
def balanced(df, n, seed=42):
    return pd.concat([df[df.label == l].sample(n=n // 2, random_state=seed) for l in (0, 1)])
sample = pd.concat([balanced(imdb[imdb.split == "test"], N_PER_DOMAIN).assign(domain="movie"),
                    balanced(amz[amz.split == "test"], N_PER_DOMAIN).assign(domain="product")], ignore_index=True)
sample = sample.sample(frac=1, random_state=0).reset_index(drop=True)
sample[["review_id", "domain", "label"]].to_csv(f"{RES}/rq4_sample_ids.csv", index=False)

def shorten(t, n):
    w = t.split(); return " ".join(w[:n]) + (" …" if len(w) > n else "")

def pick_examples(df, k, seed=7):
    """k balanced, alternating examples of moderate length from the training data."""
    d = df[(df.token_count >= 20) & (df.token_count <= 150)]
    pos = d[d.label == 1].sample(n=k // 2, random_state=seed); neg = d[d.label == 0].sample(n=k // 2, random_state=seed)
    ex = [r for pair in zip(pos.itertuples(), neg.itertuples()) for r in pair]
    return [(shorten(r.review_text, MAX_EXAMPLE_WORDS), "positive" if r.label else "negative") for r in ex]

EXAMPLES = {dom: {k: pick_examples(src, k) for k in set(SETTINGS.values()) if k}
            for dom, src in [("movie", imdb[imdb.split == "train"]), ("product", amz[amz.split == "finetune_pool"])]}
print(sample.groupby(["domain", "label"]).size())

## 3. Prompt and model wrappers
The same chat-style prompt is used for every provider. Few-shot examples are given as previous user/assistant turns.

In [ ]:
SYSTEM = ("You are a sentiment classifier for customer reviews. Read the review and answer with exactly one word: "
          "positive or negative.")

def build_messages(text, domain, k):
    kind = "movie review" if domain == "movie" else "product review"
    msgs = []
    for ex, lab in (EXAMPLES[domain][k] if k else []):
        msgs += [{"role": "user", "content": f"Classify the sentiment of this {kind}.\n\nReview: {ex}"},
                 {"role": "assistant", "content": lab}]
    msgs.append({"role": "user", "content": f"Classify the sentiment of this {kind}.\n\nReview: {shorten(text, MAX_REVIEW_WORDS)}"})
    return msgs

def parse(text):
    t = (text or "").strip().lower()
    if re.search(r"\bpositive\b", t) and not re.search(r"\bnegative\b", t): return 1
    if re.search(r"\bnegative\b", t) and not re.search(r"\bpositive\b", t): return 0
    return -1                                                     # invalid answer

def secret(name):
    try:
        from google.colab import userdata; return userdata.get(name)
    except Exception:
        return os.environ.get(name)

class LLM:
    def __init__(self, provider):
        self.provider = provider
        if provider == "hf_local":
            self.tok = AutoTokenizer.from_pretrained(HF_MODEL)
            self.model = AutoModelForCausalLM.from_pretrained(HF_MODEL, dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32).to(DEVICE).eval()
            def ids(words):
                return sorted({self.tok.encode(w, add_special_tokens=False)[0] for w in words})
            self.pos_ids, self.neg_ids = ids(["positive", "Positive", " positive", " Positive"]), ids(["negative", "Negative", " negative", " Negative"])
        elif provider == "anthropic":
            import anthropic; self.client = anthropic.Anthropic(api_key=secret("ANTHROPIC_API_KEY"))
        elif provider == "openai":
            import openai; assert OPENAI_MODEL, "Set OPENAI_MODEL"; self.client = openai.OpenAI(api_key=secret("OPENAI_API_KEY"))

    def encode(self, msgs):
        text = self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        return self.tok(text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)

    @torch.no_grad()
    def classify(self, msgs):
        """Returns (pred, prob_positive, input_tokens, output_tokens, raw_text)."""
        if self.provider == "hf_local":
            prompt = self.encode([{"role": "system", "content": SYSTEM}] + msgs)
            logits = self.model(prompt).logits[0, -1].float()
            lp, ln = torch.logsumexp(logits[self.pos_ids], 0), torch.logsumexp(logits[self.neg_ids], 0)
            prob = torch.sigmoid(lp - ln).item()
            return int(prob >= .5), prob, prompt.shape[1], 1, "positive" if prob >= .5 else "negative"
        for attempt in range(6):
            try:
                if self.provider == "anthropic":
                    r = self.client.messages.create(model=ANTHROPIC_MODEL, max_tokens=5, system=SYSTEM, messages=msgs, temperature=0)
                    text = r.content[0].text; it, ot = r.usage.input_tokens, r.usage.output_tokens
                else:
                    r = self.client.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "system", "content": SYSTEM}] + msgs,
                                                            max_completion_tokens=16)
                    text = r.choices[0].message.content; it, ot = r.usage.prompt_tokens, r.usage.completion_tokens
                p = parse(text)
                return p, (np.nan if p < 0 else float(p)), it, ot, text
            except Exception as e:
                wait = 2 ** attempt; print(f"  API error ({e.__class__.__name__}); retry in {wait}s"); time.sleep(wait)
        return -1, np.nan, 0, 0, "ERROR"

    @torch.no_grad()
    def generate(self, msgs, max_new_tokens=400):
        if self.provider == "hf_local":
            prompt = self.encode([{"role": "system", "content": "You summarise customer reviews faithfully."}] + msgs)
            out = self.model.generate(prompt, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
            return self.tok.decode(out[0, prompt.shape[1]:], skip_special_tokens=True), prompt.shape[1], out.shape[1] - prompt.shape[1]
        if self.provider == "anthropic":
            r = self.client.messages.create(model=ANTHROPIC_MODEL, max_tokens=max_new_tokens, system="You summarise customer reviews faithfully.", messages=msgs, temperature=0)
            return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens
        r = self.client.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "system", "content": "You summarise customer reviews faithfully."}] + msgs,
                                                max_completion_tokens=max_new_tokens)
        return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens

llm = LLM(PROVIDER)
print(llm.classify(build_messages("Absolutely loved it, best purchase this year.", "product", 0)))

## 4. Run the prompting experiments
Each prediction is written to a CSV file as soon as it is made, so an interrupted run can be resumed.

In [ ]:
done = pd.read_csv(PRED_CSV) if os.path.exists(PRED_CSV) else pd.DataFrame(columns=["setting", "review_id"])
done_keys = set(zip(done.setting, done.review_id))
buffer = []
for setting, k in SETTINGS.items():
    todo = sample[[(setting, r) not in done_keys for r in sample.review_id]]
    print(f"{setting}: {len(todo)} reviews to score")
    for i, r in enumerate(todo.itertuples()):
        msgs = build_messages(r.review_text, r.domain, k)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        with U.Timer() as t:
            pred, prob, it, ot, raw = llm.classify(msgs)
        buffer.append(dict(setting=setting, review_id=r.review_id, domain=r.domain, label=r.label, pred=pred, prob=prob,
                           latency_ms=1000 * t.seconds, input_tokens=it, output_tokens=ot, raw=str(raw)[:40]))
        if len(buffer) >= 50 or i == len(todo) - 1:
            U.append_csv(pd.DataFrame(buffer), PRED_CSV); buffer = []
        if (i + 1) % 250 == 0:
            print(f"   {i + 1}/{len(todo)}")
preds = pd.read_csv(PRED_CSV)
print(preds.groupby("setting").size())

## 5. Fine-tuned comparator (DistilBERT from RQ2)
The same 2,000 reviews are scored. Latency is measured one review at a time (batch size 1), the same way as the language model, on both the GPU and the CPU.

In [ ]:
bert_paths = {"movie": f"{MOD2}/distilbert_imdb_s42", "product": f"{MOD2}/distilbert_amazon_adapted_n5000_s42"}
rows = []
for dom, path in bert_paths.items():
    btok = AutoTokenizer.from_pretrained(path); bm = AutoModelForSequenceClassification.from_pretrained(path).to(DEVICE).eval()
    sub = sample[sample.domain == dom]
    for i, r in enumerate(sub.itertuples()):
        x = btok(U.mask_scores(r.review_text), truncation=True, max_length=MAX_LEN_BERT, return_tensors="pt").to(DEVICE)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        with U.Timer() as t, torch.no_grad():
            p = torch.softmax(bm(**x).logits, -1)[0, 1].item()
            if DEVICE.type == "cuda": torch.cuda.synchronize()
        rows.append(dict(review_id=r.review_id, domain=dom, label=r.label, prob=p, pred=int(p >= .5), gpu_latency_ms=1000 * t.seconds))
    # CPU latency on a subset (two threads, as in RQ1)
    bm_cpu = bm.to("cpu"); nthreads = torch.get_num_threads(); torch.set_num_threads(COSTS["cpu_threads"])
    cpu_ms = []
    for r in sub.head(100 if not U.QUICK else 5).itertuples():
        x = btok(U.mask_scores(r.review_text), truncation=True, max_length=MAX_LEN_BERT, return_tensors="pt")
        with U.Timer() as t, torch.no_grad(): bm_cpu(**x)
        cpu_ms.append(1000 * t.seconds)
    torch.set_num_threads(nthreads)
    for row in rows:
        if row["domain"] == dom: row["cpu_latency_ms"] = float(np.median(cpu_ms))
    del bm, bm_cpu; torch.cuda.empty_cache()
bert = pd.DataFrame(rows); bert.to_csv(f"{RES}/rq4_distilbert_preds.csv", index=False)
print(bert.groupby("domain")[["gpu_latency_ms", "cpu_latency_ms"]].median())

## 6. Results
### Table 16 — Accuracy, latency and cost per 1,000 reviews

In [ ]:
def cost_1k_llm(df):
    if PROVIDER == "hf_local":   # self-hosted LLM on a GPU billed per hour of use
        return df.latency_ms.mean() / 1000 * 1000 / 3600 * COSTS["gpu_usd_per_hour"]
    pr = API_PRICES[PROVIDER]
    if pr["input"] is None:
        return np.nan
    return 1000 * (df.input_tokens.mean() * pr["input"] + df.output_tokens.mean() * pr["output"]) / 1e6

rows = []
for setting in SETTINGS:
    for dom in ["movie", "product", "all"]:
        d = preds[(preds.setting == setting) & ((preds.domain == dom) | (dom == "all"))]
        valid = d.pred >= 0
        m = U.classification_metrics(d.label, d.pred.clip(lower=0) if PROVIDER != "hf_local" else d.prob)
        acc = ((d.pred == d.label)).mean()          # invalid answers count as errors
        rows.append(dict(model=f"LLM {setting.replace('_', '-')}", domain=dom, n=len(d), accuracy=acc, macro_f1=m["macro_f1"],
                         invalid_rate=1 - valid.mean(), latency_ms=d.latency_ms.median(), cost_1k_usd=cost_1k_llm(d),
                         mean_input_tokens=d.input_tokens.mean()))
for dom in ["movie", "product", "all"]:
    d = bert[(bert.domain == dom) | (dom == "all")]
    m = U.classification_metrics(d.label, d.prob)
    rows.append(dict(model="Fine-tuned DistilBERT", domain=dom, n=len(d), accuracy=m["accuracy"], macro_f1=m["macro_f1"], invalid_rate=0.0,
                     latency_ms=d.gpu_latency_ms.median(), cost_1k_usd=d.gpu_latency_ms.mean() / 1000 * 1000 / 3600 * COSTS["gpu_usd_per_hour"],
                     mean_input_tokens=np.nan, cpu_latency_ms=d.cpu_latency_ms.median(),
                     cost_1k_cpu_usd=d.cpu_latency_ms.mean() / 1000 * 1000 / 3600 * COSTS["cpu_usd_per_hour"]))
res = pd.DataFrame(rows); res.to_csv(f"{RES}/rq4_results.csv", index=False)

table16 = res[res.domain.isin(["movie", "product", "all"])].copy()
table16["accuracy"] = (100 * table16.accuracy).map("{:.2f}%".format)
table16["macro_f1"] = table16.macro_f1.map("{:.4f}".format)
table16["invalid_rate"] = (100 * table16.invalid_rate).map("{:.1f}%".format)
table16["latency_ms"] = table16.latency_ms.map("{:.1f}".format)
table16["cost_1k_usd"] = table16.cost_1k_usd.map(lambda v: "n/a (set prices)" if pd.isna(v) else f"{v:.4f}")
table16 = table16[["model", "domain", "n", "accuracy", "macro_f1", "invalid_rate", "latency_ms", "cost_1k_usd"]]
table16.columns = ["Model", "Domain", "n", "Accuracy", "Macro F1", "Invalid answers", "Median latency (ms/review)", "Cost / 1k reviews (USD)"]
U.save_table(table16, "table16_rq4_llm_vs_finetuned")
print(f"LLM: {PROVIDER} / {MODEL_TAG}; DistilBERT latency on GPU (batch 1); cost assumptions: {COSTS}")
table16

### Table 17 — Paired McNemar tests: LLM versus fine-tuned DistilBERT

In [ ]:
m = preds.merge(bert[["review_id", "pred"]].rename(columns={"pred": "bert_pred"}), on="review_id")
rows = []
for setting in SETTINGS:
    for dom in ["movie", "product", "all"]:
        d = m[(m.setting == setting) & ((m.domain == dom) | (dom == "all"))]
        r = U.mcnemar(d.label, d.bert_pred, d.pred.where(d.pred >= 0, 1 - d.label))   # invalid LLM answer = wrong
        rows.append({"LLM setting": setting, "Domain": dom, "DistilBERT right, LLM wrong": r["b"], "DistilBERT wrong, LLM right": r["c"],
                     "Disagree": f"{100 * r['disagree']:.1f}%", "Chi-square": f"{r['chi2']:.2f}", "p": U.fmt_p(r["p"])})
table17 = pd.DataFrame(rows); U.save_table(table17, "table17_rq4_mcnemar"); table17

### Figure 15 — Accuracy versus cost per 1,000 reviews

In [ ]:
allr = res[res.domain == "all"].dropna(subset=["cost_1k_usd"])
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for r in allr.itertuples():
    c = U.MODEL_COLORS["DistilBERT"] if r.model.startswith("Fine") else U.MODEL_COLORS["LLM"]
    ax.scatter(r.cost_1k_usd, 100 * r.accuracy, s=90, color=c, edgecolor="k")
    ax.annotate(f"{r.model}\n{r.latency_ms:.0f} ms/review", (r.cost_1k_usd, 100 * r.accuracy), xytext=(8, -6), textcoords="offset points", fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("Cost per 1,000 reviews (USD, log scale)"); ax.set_ylabel("Accuracy on 2,000 reviews (%)")
ax.set_title(f"Figure 15. Foundation model ({MODEL_TAG}) vs fine-tuned DistilBERT")
U.save_fig(fig, "fig15_llm_cost_accuracy"); plt.show()

## 7. Retrieval-augmented theme summaries for 30 products
For each product, all of its reviews are embedded with `all-MiniLM-L6-v2`. Two queries retrieve the eight reviews most relevant to praise and to complaints, and **retrieval is restricted to the product's own reviews**. The language model then writes three praise and three complaint themes, citing the numbered reviews that support each one.

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(DEVICE))
rag = pd.read_csv(U.P("data", "processed", "amazon_rag_products.csv"), keep_default_na=False)
products = rag.parent_asin.value_counts().index[:N_PRODUCTS]
QUERIES = {"praise": "what customers like, praise and recommend about this product",
           "complaint": "problems, defects and complaints customers have with this product"}

def summarise(asin):
    d = rag[rag.parent_asin == asin].reset_index(drop=True)
    E = st.encode(d.review_text.tolist(), normalize_embeddings=True, convert_to_numpy=True)
    picked = []
    for q in QUERIES.values():
        qv = st.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0]
        picked += list(np.argsort(-E @ qv)[:TOP_K])
    picked = list(dict.fromkeys(picked))                     # unique, in retrieval order
    ctx = "\n".join(f"[R{i + 1}] ({d.rating[j]}★) {shorten(d.review_text[j], 120)}" for i, j in enumerate(picked))
    prompt = (f"Below are {len(picked)} customer reviews of one product.\n\n{ctx}\n\n"
              "Write a summary for a product manager with two sections:\n"
              "PRAISE: the 3 main things customers like.\nCOMPLAINTS: the 3 main problems customers report.\n"
              "Each theme must be one short line and must cite the supporting reviews like [R2, R5]. "
              "Use only information stated in the reviews.")
    with U.Timer() as t:
        text, it, ot = llm.generate([{"role": "user", "content": prompt}])
    cited = [int(c) for c in re.findall(r"R(\d+)", text)]
    valid = [c for c in cited if 1 <= c <= len(picked)]
    # grounding proxy: similarity between each theme line and the reviews it cites
    sims = []
    for line in [l for l in text.splitlines() if re.search(r"R\d+", l)]:
        ids = [int(c) - 1 for c in re.findall(r"R(\d+)", line) if 1 <= int(c) <= len(picked)]
        if ids:
            lv = st.encode([re.sub(r"\[.*?\]", "", line)], normalize_embeddings=True, convert_to_numpy=True)[0]
            sims.append(float(max(E[picked[i]] @ lv for i in ids)))
    return dict(parent_asin=asin, n_reviews=len(d), mean_rating=d.rating.mean(), retrieved=len(picked), summary=text.strip(),
                citations=len(cited), invalid_citations=len(cited) - len(valid), grounding_similarity=np.mean(sims) if sims else np.nan,
                latency_s=t.seconds, input_tokens=it, output_tokens=ot, context=ctx)

summ = pd.DataFrame([summarise(a) for a in products])
summ.to_csv(f"{RES}/rq4_rag_summaries.csv", index=False)
print(f"{len(summ)} summaries | mean citations {summ.citations.mean():.1f} | invalid citations {summ.invalid_citations.sum()} | "
      f"mean grounding similarity {summ.grounding_similarity.mean():.3f} | mean latency {summ.latency_s.mean():.1f}s")
print("\nExample summary (product", summ.parent_asin[0], "):\n", summ.summary[0])

### Human rating sheet (faithfulness and usefulness)
The cell below creates `outputs/results/rq4_summary_ratings.csv`, with one row per product per rater. Each rater reads the summary and the retrieved reviews (`context` column in `rq4_rag_summaries.csv`) and scores:
* **Faithfulness:** 3 = every theme is supported by the cited reviews; 2 = minor unsupported detail; 1 = at least one theme is unsupported or wrong.
* **Usefulness:** 3 = actionable for a product manager; 2 = partly useful; 1 = generic or not useful.

Fill in the sheet in Google Sheets or Excel, save it as CSV in the same place, and run the next cell to compute the mean scores and the inter-rater agreement.

In [ ]:
RATINGS = f"{RES}/rq4_summary_ratings.csv"
if not os.path.exists(RATINGS):
    sheet = pd.DataFrame([dict(parent_asin=a, rater=r, faithfulness="", usefulness="", comment="") for a in summ.parent_asin for r in ["author", "peer"]])
    sheet.to_csv(RATINGS, index=False); print("Rating sheet created:", RATINGS)

In [ ]:
from sklearn.metrics import cohen_kappa_score
rt = pd.read_csv(RATINGS)
if rt[["faithfulness", "usefulness"]].notna().all().all() and len(rt):
    rows = []
    for c in ["faithfulness", "usefulness"]:
        w = rt.pivot(index="parent_asin", columns="rater", values=c)
        rows.append({"Criterion": c.capitalize(), "Mean (1-3)": f"{rt[c].mean():.2f}", "% rated 3": f"{100 * (rt[c] == 3).mean():.0f}%",
                     "Weighted kappa (author vs peer)": f"{cohen_kappa_score(w['author'], w['peer'], weights='linear'):.2f}"})
    table18 = pd.DataFrame(rows); U.save_table(table18, "table18_rq4_summary_ratings"); display(table18)
else:
    print("Ratings not filled in yet; complete", RATINGS, "and re-run this cell.")

## 8. Break-even analysis
Monthly cost as a function of review volume *V*:
* **Pay-per-call foundation model:** `V × c_LLM`, where `c_LLM` is the cost per review from Table 16.
* **Self-hosted fine-tuned DistilBERT:** a fixed monthly cost `F = GPU price per hour × 730 hours` for one always-on instance. Its capacity is `730 × 3600 / seconds per review`, and an extra instance is added for each multiple of that capacity. The one-off cost of fine-tuning (GPU time plus labelling) is spread over 12 months.

**Break-even volume:** `V* = (F + one-off / 12) / c_LLM`. The formula is printed below so that it can be recomputed with new prices.

In [ ]:
LABEL_COST_USD = 0.05                     # assumed cost per labeled review (update)
rq2 = json.load(open(f"{RES}/rq2_decision.json"))
src_runs = pd.read_csv(f"{RES}/rq2_source_runs.csv")
labels_used = max([v for v in rq2["labels_needed_in_domain"].values() if v] or [5000])
one_off = src_runs.train_time_min.mean() / 60 * COSTS["gpu_usd_per_hour"] + labels_used * LABEL_COST_USD
F = COSTS["gpu_usd_per_hour"] * 730
sec_per_review = bert.gpu_latency_ms.mean() / 1000
capacity = 730 * 3600 / sec_per_review

llm_rows = res[(res.domain == "all") & res.model.str.startswith("LLM")].dropna(subset=["cost_1k_usd"])
if llm_rows.empty:
    print("Set API_PRICES to compute the break-even volume.")
else:
    best_llm = llm_rows.sort_values("accuracy", ascending=False).iloc[0]
    c_llm = best_llm.cost_1k_usd / 1000
    V_star = (F + one_off / 12) / c_llm
    print(f"Best LLM setting: {best_llm.model} ({100 * best_llm.accuracy:.2f}%), cost per review ${c_llm:.6f}")
    print(f"Self-hosted DistilBERT: F = ${F:.0f}/month, capacity ≈ {capacity / 1e6:.1f} M reviews/month per GPU, one-off ${one_off:.0f}")
    print(f"BREAK-EVEN: V* = (F + one-off/12) / c_LLM = ({F:.0f} + {one_off / 12:.1f}) / {c_llm:.6f} ≈ {V_star:,.0f} reviews per month")

    V = np.logspace(3, 9, 200)
    cost_llm = V * c_llm
    cost_bert = np.ceil(V / capacity) * F + one_off / 12
    cost_lr = np.ceil(V / (730 * 3600 / (pd.read_csv(f"{RES}/rq1_latency.csv").set_index("model").loc["logreg", "cpu_ms_per_review"] / 1000))) * COSTS["cpu_usd_per_hour"] * 730
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(V, cost_llm, label=f"Pay-per-call LLM ({best_llm.model})", color=U.MODEL_COLORS["LLM"])
    ax.plot(V, cost_bert, label="Self-hosted DistilBERT (GPU)", color=U.MODEL_COLORS["DistilBERT"])
    ax.plot(V, cost_lr, label="Self-hosted TF-IDF + LogReg (CPU)", color=U.MODEL_COLORS["TF-IDF + LogReg"], ls="--")
    ax.axvline(V_star, color="k", ls=":", lw=1)
    ax.annotate(f"break-even\n≈ {V_star:,.0f}/month", (V_star, 0.85), xycoords=("data", "axes fraction"), xytext=(6, 0), textcoords="offset points", fontsize=8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Reviews per month (log scale)"); ax.set_ylabel("Monthly cost (USD, log scale)")
    ax.set_title("Figure 16. Monthly cost by review volume"); ax.legend(fontsize=8)
    U.save_fig(fig, "fig16_break_even"); plt.show()
    json.dump(dict(provider=PROVIDER, model=MODEL_TAG, run_date=time.strftime("%Y-%m-%d"), best_llm_setting=best_llm.model,
                   best_llm_accuracy=float(best_llm.accuracy), c_llm_per_review=float(c_llm), fixed_monthly_gpu=F, one_off=one_off,
                   capacity_per_gpu_month=capacity, break_even_reviews_per_month=float(V_star), label_cost_usd=LABEL_COST_USD),
              open(f"{RES}/rq4_decision.json", "w"), indent=2)

## Summary for the report
* **Table 16 / Figure 15:** accuracy, latency and cost of zero-shot and few-shot prompting against fine-tuned DistilBERT on the same 2,000 reviews.
* **Table 17:** paired McNemar tests.
* **RAG summaries:** `rq4_rag_summaries.csv` (30 products) with automatic citation and grounding checks. Table 18 follows once the two raters have filled in the rating sheet.
* **Figure 16 / `rq4_decision.json`:** the break-even volume and the formula behind it.

**Next:** `07_Final_Summary.ipynb` combines all four research questions into the tiered architecture recommendation.